01_text_preprocess (2).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/16Q_ZTihtx8ki8oObafB_GvnylEXAHnc1

# DeepTrace — Module 2: Text/Scam Preprocessing
## Notebook 1 of 2 | CPU Runtime | ~15 min

**Datasets (all auto-downloaded except Mendeley which is optional):**
- HuggingFace: `ealvaradob/phishing-dataset`
- HuggingFace: `SetFit/enron_spam`
- Kaggle: SMS Spam Collection
- Kaggle: Phishing Email Dataset
- Kaggle: Phishing and Legitimate Emails

In [ ]:
# CELL 1 — Install
!pip install -q kaggle datasets transformers torch pandas scikit-learn

In [ ]:
# CELL 2 — Kaggle credentials
from google.colab import files
import os

print('Upload kaggle.json')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
fname = next(iter(uploaded.keys()))
with open('/root/.kaggle/kaggle.json','wb') as f:
    f.write(uploaded[fname])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✓ Kaggle set')

In [ ]:
# CELL 3 — HuggingFace datasets
import pandas as pd, os
from datasets import load_dataset

os.makedirs('/content/data/raw', exist_ok=True)
os.makedirs('/content/data/processed', exist_ok=True)
all_data = []

# ─── ealvaradob/phishing-dataset ─────────────────────────────
print('Loading ealvaradob/phishing-dataset...')
try:
    ds = load_dataset('ealvaradob/phishing-dataset', split='train')
    df = ds.to_pandas()
    print(f'  Columns: {list(df.columns)}')
    text_col  = next((c for c in df.columns if 'text' in c.lower()), df.columns[0])
    label_col = next((c for c in df.columns if 'label' in c.lower() or 'class' in c.lower()), df.columns[-1])
    sub = df[[text_col,label_col]].dropna().copy()
    sub.columns = ['text','label']
    sub['label'] = pd.to_numeric(sub['label'], errors='coerce').fillna(0).astype(int)
    sub = sub[sub['label'].isin([0,1])]
    all_data.append(sub)
    print(f'  ✓ {len(sub)} rows  labels={sub.label.value_counts().to_dict()}')
except Exception as e:
    print(f'  ⚠ {e}')

# ─── SetFit/enron_spam ────────────────────────────────────────
print('Loading SetFit/enron_spam...')
try:
    ds2 = load_dataset('SetFit/enron_spam', split='train')
    df2 = ds2.to_pandas()
    print(f'  Columns: {list(df2.columns)}')
    text_col  = 'text'  if 'text'  in df2.columns else df2.columns[0]
    label_col = 'label' if 'label' in df2.columns else df2.columns[-1]
    sub2 = df2[[text_col,label_col]].dropna().copy()
    sub2.columns = ['text','label']
    sub2['label'] = pd.to_numeric(sub2['label'], errors='coerce').fillna(0).astype(int)
    sub2 = sub2[sub2['label'].isin([0,1])]
    all_data.append(sub2)
    print(f'  ✓ {len(sub2)} rows  labels={sub2.label.value_counts().to_dict()}')
except Exception as e:
    print(f'  ⚠ {e}')

In [ ]:
# CELL 4 — Kaggle datasets
import subprocess, glob

kaggle_sets = [
    ('uciml/sms-spam-collection-dataset',              'sms_spam'),
    ('naserabdullahalam/phishing-email-dataset',       'phishing_emails'),
    ('kuladeep19/phishing-and-legitimate-emails-dataset','kuladeep_emails'),
]

for dataset_id, folder in kaggle_sets:
    dest = f'/content/data/raw/{folder}'
    os.makedirs(dest, exist_ok=True)
    r = subprocess.run(['kaggle','datasets','download','-d',dataset_id,'--unzip','-p',dest,'-q'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        csvs = glob.glob(f'{dest}/**/*.csv', recursive=True)
        print(f'✓ {folder}: {len(csvs)} CSV(s)')
    else:
        print(f'⚠ {folder}: {r.stderr[:100]}')

In [ ]:
# CELL 5 — Parse Kaggle CSVs
import glob

def try_load_csv(path, text_hints, label_hints, pos_vals):
    for enc in ['utf-8','latin-1','cp1252']:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            df.columns = df.columns.str.lower().str.strip()
            tc = next((c for c in df.columns if any(h in c for h in text_hints)), None)
            lc = next((c for c in df.columns if any(h in c for h in label_hints)), None)
            if tc and lc:
                sub = df[[tc,lc]].dropna().copy()
                sub.columns = ['text','label']
                sub['label'] = sub['label'].astype(str).str.lower()
                sub['label'] = sub['label'].apply(lambda x: 1 if any(v in x for v in pos_vals) else 0)
                return sub
        except: continue
    return None

# SMS Spam
for f in glob.glob('/content/data/raw/sms_spam/**', recursive=True):
    if f.endswith('.csv'):
        df = try_load_csv(f,['text','message','sms','v2'],['label','class','v1','category'],['spam','1'])
        if df is not None:
            all_data.append(df)
            print(f'✓ SMS Spam: {len(df)} rows  labels={df.label.value_counts().to_dict()}')

# Phishing Emails
for f in glob.glob('/content/data/raw/phishing_emails/**', recursive=True):
    if f.endswith('.csv'):
        df = try_load_csv(f,['text','email','body','message','content'],['label','class','type','category'],['phishing','spam','1'])
        if df is not None:
            all_data.append(df)
            print(f'✓ Phishing Emails: {len(df)} rows  labels={df.label.value_counts().to_dict()}')

# Kuladeep
for f in glob.glob('/content/data/raw/kuladeep_emails/**', recursive=True):
    if f.endswith('.csv'):
        df = try_load_csv(f,['text','email','body','message','content'],['label','class','type','category'],['phishing','spam','1'])
        if df is not None:
            all_data.append(df)
            print(f'✓ Kuladeep Emails: {len(df)} rows  labels={df.label.value_counts().to_dict()}')

print(f'\nTotal sources loaded: {len(all_data)}')

In [ ]:
# CELL 6 — Optional Mendeley upload (skip if not available)
print('Optional: Upload Mendeley CSV files (or skip/cancel)')
try:
    mendeley = files.upload()
    for fname, content in mendeley.items():
        out_path = f'/content/data/raw/{fname}'
        with open(out_path,'wb') as f: f.write(content)
        df = try_load_csv(out_path,['text','message','sms','content'],
                          ['label','class','type','category'],
                          ['spam','smishing','phishing','1'])
        if df is not None:
            all_data.append(df)
            print(f'✓ Mendeley {fname}: {len(df)} rows  labels={df.label.value_counts().to_dict()}')
except:
    print('Skipping Mendeley')

In [ ]:
# CELL 7 — Combine, clean, balance
import re
from collections import Counter
from sklearn.model_selection import train_test_split

assert len(all_data) > 0, 'No data loaded! Check previous cells.'

full = pd.concat(all_data, ignore_index=True)
full = full.dropna(subset=['text','label'])
full['text']  = full['text'].astype(str).str.strip()
full['label'] = pd.to_numeric(full['label'], errors='coerce').fillna(0).astype(int)
full = full[full['label'].isin([0,1])]
full = full[full['text'].str.len() >= 10]

# Clean text
def clean(t):
    t = re.sub(r'\s+',' ', str(t)).strip()
    t = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f-\x9f]','',t)
    return t[:1024]

full['text'] = full['text'].apply(clean)
full = full.drop_duplicates(subset='text').reset_index(drop=True)

print(f'Total samples: {len(full)}')
print(f'Label dist: {Counter(full.label)}')
print(f'Phishing ratio: {full.label.mean():.2%}')

# IMPORTANT: balance to max 1:2 ratio (phish:legit) so the model can learn both classes
n_phish = full[full.label==1].shape[0]
n_legit = full[full.label==0].shape[0]
max_legit = n_phish * 2  # at most 2x as many legit as phish

if n_legit > max_legit:
    legit_sub = full[full.label==0].sample(n=max_legit, random_state=42)
    full = pd.concat([full[full.label==1], legit_sub]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'After balancing: {Counter(full.label)}')

# Split: 70% train, 15% val, 15% test
tr, te = train_test_split(full, test_size=0.15, stratify=full.label, random_state=42)
tr, va = train_test_split(tr,   test_size=0.1765, stratify=tr.label, random_state=42)

print(f'\nTrain: {len(tr)}  Val: {len(va)}  Test: {len(te)}')
print(f'Train labels: {Counter(tr.label)}')
print(f'Val labels:   {Counter(va.label)}')
print(f'Test labels:  {Counter(te.label)}')

In [ ]:
# CELL 8 — Save CSVs
tr.to_csv('/content/data/processed/text_train.csv', index=False)
va.to_csv('/content/data/processed/text_val.csv',   index=False)
te.to_csv('/content/data/processed/text_test.csv',  index=False)

print('✓ Saved')
!ls -lh /content/data/processed/

# Sanity check — make sure both labels are present in train
assert set(tr.label.unique()) == {0,1}, 'Missing a class in training data!'
print('✓ Both classes present in training set')

In [ ]:
# CELL 9 — Download
import shutil
from google.colab import files

shutil.make_archive('/content/text_processed','zip','/content/data/processed')
files.download('/content/text_processed.zip')
print('✅ Upload this zip to 02_text_train.ipynb')